# Foundations of scalar optics

**What does scalar optics describe, and how does it help us study diffraction?**

This notebook is a **preview of the field**. Its central quantity is the complex scalar field $U(\mathbf r)$: one function containing the amplitude and spatial phase of monochromatic light. The wave and Helmholtz equations constrain its possible forms; the experiment selects a solution through its sources and boundary conditions. Diffraction is one application: find how an aperture's field develops after propagation.

**First reading:** follow the central idea of each section, compare the solution shapes in §5, and inspect the aperture example in §8. The derivative checks, detailed solution families, and small calculations are optional depth to revisit. You do not need to master them before Notebook 1.

**Experimental thread:** connect each model quantity to what an apparatus records and what can be inferred from that record. The framework should help us predict a measurement and interpret it.

The route is **complex scalar field $U$ → Helmholtz → several solution families → boundary conditions → diffraction**. We then choose **plane waves as useful building blocks** and learn their geometry gradually in Notebooks 1–5, followed by amplitude, intensity, and superposition in Notebooks 6–7.

Run the code cells in order with NumPy, SciPy, Matplotlib, Plotly, and ipywidgets. Each explorer has its own controls and reset button; the editable calculations run separately. We use $e^{-i\omega t}$, wavelengths **in the medium**, and positive $z$ for forward propagation. The aperture example explicitly uses a far-field approximation.

In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import importlib
import numpy as np
from IPython.display import display

if 'scalar_theory_lessons' in globals():
    for old_lesson in scalar_theory_lessons.values():
        old_lesson.close()

# Refresh shared helpers before the views that import them.
for _module_name in (
    'wave_optics._view_cache',
    'wave_optics.waves',
    'wave_optics.diffraction',
    'wave_optics.plotting',
    'wave_optics.grating_intro',
    'wave_optics.scalar_theory',
    'wave_optics.helmholtz_families',
    'wave_optics.scalar_theory_view',
):
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])
    else:
        importlib.import_module(_module_name)

import wave_optics.scalar_theory as scalar_theory
import wave_optics.helmholtz_families as helmholtz_families
import wave_optics.scalar_theory_view as scalar_theory_view
from wave_optics.scalar_theory_view import ScalarTheoryExplorer

scalar_theory_lessons = {}

def show_scalar_theory(stage):
    stage = 'solutions' if stage == 'more_solutions' else stage
    if stage in scalar_theory_lessons:
        scalar_theory_lessons[stage].close()
    lesson = ScalarTheoryExplorer(stage)
    scalar_theory_lessons[stage] = lesson
    display(lesson.ui)

## 1. The central quantity: a complex scalar field

A scalar field $u(\mathbf r,t)$ assigns one signed value to each position and time. In scalar optics, this represents an optical field component or a scalar approximation to the field. **Scalar does not mean plane wave:** its wavefronts may be planar, curved, or more complicated.

We consider a linear, homogeneous, isotropic, nonabsorbing medium and a propagation region containing no sources. A laser may supply light from outside that region. For the time-domain equation we use a constant wave speed $v=c/n$, a nondispersive model.

Maxwell's equations motivate this model: under these material assumptions, each Cartesian field component satisfies a scalar wave equation. The full electromagnetic field must also satisfy vector constraints and interface conditions. A scalar treatment is appropriate when the polarization details being omitted do not control the result. [TU Delft, *Basic Electromagnetic and Wave Optics*, Eqs. (28)–(32)](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap1_Basics/Basics_2022_01Clean.html).

For monochromatic light, the central quantity is the **complex scalar field** $U(\mathbf r)$. It stores amplitude and spatial phase at each position. Euler's identity gives the connection:

$$e^{i\phi}=\cos\phi+i\sin\phi,\qquad U=Ae^{i\phi}.$$

The corresponding real, time-dependent field is

$$u(\mathbf r,t)=\operatorname{Re}\{U(\mathbf r)e^{-i\omega t}\}
=A(\mathbf r)\cos[\phi(\mathbf r)-\omega t].$$

The field $U$ has no time dependence; $u$ is the real oscillation recovered from it. This complex representation is also called a **phasor representation**. We will return to that viewpoint when it simplifies propagation and interference calculations.

**First-reading takeaway:** scalar diffraction predicts $U$ after propagation, then uses $|U|^2$ to obtain a relative intensity. The algebra below shows why the square appears.

The real field oscillates; a detector averages over many optical cycles. Our relative scalar intensity convention is

$$\langle u^2\rangle_T
=A^2\left\langle\frac{1+\cos(2\phi-2\omega t)}{2}\right\rangle_T
=\frac{A^2}{2}=\frac{|U|^2}{2},$$

$$\frac{I}{I_{\rm ref}}
=\frac{\langle u^2\rangle_T}{A_{\rm ref}^2/2}
=\frac{|U|^2}{A_{\rm ref}^2}.$$

We use $A_{\rm ref}=1$. This measures local mean-square field in the scalar model; it is not a vector energy-flux calculation. A standing wave, for example, can have nonzero mean-square field while its net energy transport vanishes.

### Experimental connection: what is recorded, and what is inferred?

An ordinary camera records a signal related to the light incident on each pixel,
integrated over its area and exposure time. Calibration connects that signal to
an intensity; it does not give an instantaneous signed field value.
The distinction between field and detected intensity is developed in the
[TU Delft interference chapter](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap5_Interference/InterferenceCoherence_2022_01Clean.html#interference-of-monochromatic-fields-of-the-same-frequency).

| Experimental information | What the model lets us infer |
| --- | --- |
| Intensity ratio for the same travelling-wave geometry | Amplitude ratio $|U|/A_{\rm ref}=\sqrt{I/I_{\rm ref}}$ |
| Interference with a known reference, with enough phase information | The field's phase relative to that reference |
| Wavelength in the medium and propagation direction | The plane-wave vector $\mathbf k$ |
| A diffraction intensity profile | A test of the predicted propagated field through $|U|^2$ |

One intensity image generally does **not** determine the full complex $U$:
it lacks phase information. We predict or reconstruct $U$, then compare its
observable consequences with the experiment. Phase is accessible through an
appropriate measurement, rather than intrinsically unmeasurable.


## 2. The wave equation connects space and time

**Preview:** an allowed wave must have compatible variation in space and time. The following derivative check makes that constraint explicit.

In the chosen medium,

$$\boxed{\nabla^2u-\frac{1}{v^2}\frac{\partial^2u}{\partial t^2}=0,}
\qquad
\nabla^2u=\frac{\partial^2u}{\partial x^2}
+\frac{\partial^2u}{\partial y^2}
+\frac{\partial^2u}{\partial z^2}.$$

The spatial curvature and temporal acceleration must balance. In one dimension, try $u=A\cos(qx-\omega t)$:

$$\frac{\partial u}{\partial x}=-Aq\sin(qx-\omega t),
\qquad
\frac{\partial^2u}{\partial x^2}=-Aq^2\cos(qx-\omega t)=-q^2u,$$

$$\frac{\partial u}{\partial t}=A\omega\sin(qx-\omega t),
\qquad
\frac{\partial^2u}{\partial t^2}=-A\omega^2\cos(qx-\omega t)=-\omega^2u.$$

Substitution gives

$$\frac{\partial^2u}{\partial x^2}
-\frac{1}{v^2}\frac{\partial^2u}{\partial t^2}
=\left(\frac{\omega^2}{v^2}-q^2\right)u.$$

For a nonzero wave this vanishes everywhere only when $q^2=\omega^2/v^2$. The spatial oscillation rate and temporal frequency cannot be chosen independently once the medium is fixed.

## 3. Separate the monochromatic time dependence

Write $u_c(\mathbf r,t)=U(\mathbf r)e^{-i\omega t}$ for the complex expression including time, with $u=\operatorname{Re}u_c$. The spatial field is still $U$. Its derivatives are

$$\frac{\partial u_c}{\partial t}
=-i\omega Ue^{-i\omega t},\qquad
\frac{\partial^2u_c}{\partial t^2}
=(-i\omega)^2Ue^{-i\omega t}=-\omega^2Ue^{-i\omega t},$$

$$\nabla^2u_c=e^{-i\omega t}\nabla^2U.$$

The wave equation becomes

$$e^{-i\omega t}\nabla^2U
-\frac{1}{v^2}\left(-\omega^2Ue^{-i\omega t}\right)
=e^{-i\omega t}\left(\nabla^2U+\frac{\omega^2}{v^2}U\right)=0.$$

Since the exponential never vanishes,

$$\boxed{\nabla^2U+k^2U=0,}
\qquad k=\frac{\omega}{v}=\frac{2\pi}{\lambda}.$$

This is the **scalar Helmholtz equation**. It constrains the complex scalar field $U$ at one frequency. Recovering $u$ still requires the time factor.

With $v=c/n$ and $\lambda_0=2\pi c/\omega$,

$$k=\frac{n\omega}{c}=\frac{2\pi n}{\lambda_0},
\qquad
\lambda=\frac{2\pi}{k}=\frac{\lambda_0}{n}.$$

For a dispersive material, a monochromatic calculation uses the refractive index at that frequency. Here it is real because absorption is excluded.

### Small calculation 1: wavelength in a medium

For vacuum wavelength $0.6\,\mu\mathrm m$ and $n=1.5$, predict the medium wavelength. Lengths below are in micrometres and times in femtoseconds.

In [ ]:
lambda0_um = 0.6
n = 1.5
c_um_per_fs = 0.299792458
omega = 2*np.pi*c_um_per_fs/lambda0_um
v = c_um_per_fs/n
lambda_um = lambda0_um/n
k = omega/v
print(f'Wavelength in medium = {lambda_um:.3f} um')
print(f'k = omega/v = {k:.6f} rad/um')
print(f'2 pi/lambda = {2*np.pi/lambda_um:.6f} rad/um')

<details><summary>Answer</summary>

The medium wavelength is $0.6/1.5=0.4\,\mu\mathrm m$, and both routes give $k=5\pi\approx15.708\,\mathrm{rad}/\mu\mathrm m$. The frequency remains fixed; the medium changes the phase speed and wavelength.

</details>

## 4. Optional worked check: does a trial field satisfy Helmholtz?

Fix the physical wavenumber $k$ and test $U(x)=Ae^{i(qx+\phi)}$:

$$U'=iqU,\qquad U''=(iq)^2U=-q^2U,$$

$$R(x)\equiv U''+k^2U=(k^2-q^2)U.$$

$R$ is the **residual**: a nonzero residual means this trial field is not a solution at the chosen frequency and medium. For $q>0$, it vanishes when $q=k$. The field with $q\ne k$ could belong to another frequency, but not the one currently fixed.

The left graph compares $\operatorname{Re}U$ with $\operatorname{Re}(-U''/k^2)$; Helmholtz requires agreement. The right graph displays $\operatorname{Re}(R/k^2)$. These derivatives are evaluated analytically.

**Try:** keep a nonzero amplitude, set **Trial q/k** to 0.5, 1, and 1.5, and compare the curves. Changing phase slides the pattern but cannot repair the wrong spatial frequency. A zero amplitude gives the trivial zero solution for any trial $q$.

In [ ]:
show_scalar_theory('helmholtz')

### Small calculation 2: quantify the mismatch

The normalized residual is $R/k^2=[1-(q/k)^2]U$. For unit amplitude, predict its maximum magnitude for the three trial ratios.

In [ ]:
wavelength_um = 2.0
k = 2*np.pi/wavelength_um
x = np.linspace(-2, 2, 501)
amplitude, phase_deg = 1.0, 30.0
print('q/k    max |R/k^2|')
for ratio in (0.5, 1.0, 1.5):
    q = ratio*k
    U = amplitude*np.exp(1j*(q*x+np.deg2rad(phase_deg)))
    second_derivative = -q*q*U
    residual = second_derivative+k*k*U
    print(f'{ratio:3.1f}       {np.max(abs(residual))/k**2:.6f}')

<details><summary>Answer</summary>

The magnitudes are $|1-0.5^2|=0.75$, $0$, and $|1-1.5^2|=1.25$. A residual may vanish at isolated positions without vanishing as a field; the equation must hold throughout the source-free region.

</details>

## 5. Helmholtz allows more than plane waves

**Preview:** compare the different shapes of $U$ below. Plane waves are one family, rather than the definition of scalar theory. Standing, spherical, cylindrical, and structured beam fields are also useful. The usual Gaussian formulas require a paraxial approximation.

For a first reading, follow the shape and physical setting of each family and try the combined explorer. The derivative checks and special-function formulas are optional detail.

### 5.1 Plane waves

For $U=Ae^{i\mathbf k\cdot\mathbf r}$, differentiating in each coordinate gives

$$\frac{\partial^2U}{\partial x^2}=-k_x^2U,\quad
\frac{\partial^2U}{\partial y^2}=-k_y^2U,\quad
\frac{\partial^2U}{\partial z^2}=-k_z^2U,$$

$$\nabla^2U+k^2U
=\left[k^2-(k_x^2+k_y^2+k_z^2)\right]U.$$

For a propagating plane wave with real components,

$$\boxed{k_x^2+k_y^2+k_z^2=k^2.}$$

Its direction can vary while its wavenumber magnitude stays fixed.

### 5.2 Standing waves

Add equal plane waves travelling in opposite directions:

$$U(z)=\frac A2e^{ikz}+\frac A2e^{-ikz}=A\cos(kz),$$

$$U''=-k^2A\cos(kz)=-k^2U,\qquad
u(z,t)=A\cos(kz)\cos(\omega t).$$

The nodes stay at fixed positions. This is a scalar Helmholtz solution even though the total field has no single travelling-wave direction.

### 5.3 Outgoing spherical waves

For $r>0$, consider $U(r)=A_s e^{ikr}/r$. For a spherically symmetric field, the three-dimensional Laplacian is

$$\nabla^2U=\frac{1}{r^2}\frac{d}{dr}\left(r^2\frac{dU}{dr}\right).$$

Direct differentiation gives

$$\frac{dU}{dr}=A_se^{ikr}\left(\frac{ik}{r}-\frac1{r^2}\right),
\qquad r^2\frac{dU}{dr}=A_se^{ikr}(ikr-1),$$

$$\begin{aligned}
\frac{d}{dr}\left(r^2\frac{dU}{dr}\right)
&=A_se^{ikr}\left[ik(ikr-1)+ik\right]\\
&=-k^2A_sr e^{ikr},\\
\nabla^2U&=-k^2A_s\frac{e^{ikr}}r=-k^2U.
\end{aligned}$$

Its phase $kr-\omega t$ advances outward. The origin is singular and contains the ideal point source; the homogeneous equation applies away from it.

### Small calculation 3: verify the radial operator

For $U=e^{ikr}/r$, compare the full radial Laplacian with $-k^2U$. Using only $U''$ would omit a necessary geometric term.

In [ ]:
k = np.pi
r = np.array([1.0, 2.0, 4.0])  # um; never evaluate the source at r=0
phase_factor = np.exp(1j*k*r)
U = phase_factor/r
dU = phase_factor*(1j*k/r-1/r**2)
d2U = phase_factor*(-k**2/r-2j*k/r**2+2/r**3)
radial_laplacian = d2U+2*dU/r
normalized_residual = (radial_laplacian+k**2*U)/(k**2*abs(U))
print('r (um)   |U|       |U|^2       |normalized residual|')
for row in zip(r, abs(U), abs(U)**2, abs(normalized_residual)):
    print(f'{row[0]:4.1f}     {row[1]:.3f}      {row[2]:.4f}        {row[3]:.2e}')

<details><summary>Answer</summary>

The amplitudes are $1,1/2,1/4$ and their squares are $1,1/4,1/16$. The normalized residual is zero up to numerical rounding. The amplitude falls as $1/r$, while spherical area grows as $r^2$.

</details>

### 5.4 Cylindrical waves

A line source parallel to $y$ gives cylindrical symmetry, with $\rho=\sqrt{x^2+z^2}$. Its outgoing scalar field is

$$U(\rho)=A H_0^{(1)}(k\rho),\qquad
U''+\frac1\rho U'+k^2U=0\quad(\rho>0).$$

$H_0^{(1)}$ is the outgoing Hankel function. The source axis is singular. Unlike spherical spreading, cylindrical spreading has an amplitude envelope proportional to $1/\sqrt{\rho}$ at large distance:

$$H_0^{(1)}(k\rho)\sim
\sqrt{\frac{2}{\pi k\rho}}\,e^{i(k\rho-\pi/4)}
\qquad(k\rho\gg1).$$

This is an asymptotic form; the Hankel function is the exact solution. [NIST DLMF, §10.17](https://dlmf.nist.gov/10.17).

### 5.5 Bessel beams

Around the $z$ axis, now use $\rho=\sqrt{x^2+y^2}$ and

$$U(\rho,z)=A J_0(k_\perp\rho)e^{ik_z z},\qquad
0<k_\perp<k,\quad k_z=\sqrt{k^2-k_\perp^2}>0.$$

The transverse Bessel factor satisfies $f''+f'/\rho=-k_\perp^2f$. Therefore

$$\nabla^2U=-k_\perp^2U-k_z^2U,
\qquad
(\nabla^2+k^2)U=[k^2-k_\perp^2-k_z^2]U=0$$

when $k_\perp^2+k_z^2=k^2$. A cone of plane waves with these components produces the same field.

Its ideal transverse intensity is independent of $z$. The infinite transverse extent requires infinite total power; finite apertures only approximate that behavior over a finite distance. [J. Durnin, *Exact solutions for nondiffracting beams. I. The scalar theory* (1987)](https://www.durnin.info/durnin_josa_1987.pdf).

### 5.6 Spherical multipoles

Spherical waves can also have angular structure. Separate it as

$$U=R_\ell(r)Y_{\ell m}(\vartheta,\varphi).$$

The angular function supplies the eigenvalue $-\ell(\ell+1)$, leaving

$$R_\ell''+\frac2rR_\ell'
+\left[k^2-\frac{\ell(\ell+1)}{r^2}\right]R_\ell=0.$$

$j_\ell(kr)$ is regular at the origin; $h_\ell^{(1)}(kr)$ gives outgoing behavior and is singular there. Boundary conditions select the combination. The indices $\ell,m$ describe the angular pattern. [NIST DLMF, §10.73](https://dlmf.nist.gov/10.73).

The illustration uses the **regular**, axisymmetric case

$$U\propto j_\ell(kr)P_\ell(z/r),\qquad m=0,$$

with $P_0(s)=1$, $P_1(s)=s$, and $P_2(s)=(3s^2-1)/2$. Changing $\ell$ introduces angular nodes; these are not the outgoing point-source fields of §5.3.

### 5.7 Evanescent waves

For $q>k$, define $\alpha=\sqrt{q^2-k^2}$. Then

$$U=e^{iqx-\alpha z},\qquad
\nabla^2U=(-q^2+\alpha^2)U=-k^2U.$$

This is exact in the chosen half-space $z\geq0$ and decays away from its boundary. Its complex wavevector is $\mathbf k=(q,0,i\alpha)$:

$$\mathbf k\cdot\mathbf k=q^2+(i\alpha)^2=q^2-\alpha^2=k^2.$$

This dot product does **not** use complex conjugation; the sum of component magnitudes squared would be wrong.

### Small calculation 3b: build a Bessel beam from plane waves

Average equal plane-wave contributions around a cone:

$$\frac1{2\pi}\int_0^{2\pi}
e^{ik_\perp(x\cos\gamma+y\sin\gamma)}e^{ik_z z}\,d\gamma
=J_0(k_\perp\rho)e^{ik_z z}.$$

Each contribution has $k_\perp^2+k_z^2=k^2$. The code forms a finite angular sum, checks its Helmholtz residual, and compares amplitudes before and after propagation. [NIST DLMF, Bessel integral representations](https://dlmf.nist.gov/10.9).

In [ ]:
wavelength_um, transverse_ratio = 2.0, 0.6
k = 2*np.pi/wavelength_um
k_perp = transverse_ratio*k
k_z = np.sqrt(k**2-k_perp**2)
rho = np.array([0.0, 0.5, 1.0, 2.0, 3.0])  # samples on +x, with y=0
gamma = np.linspace(0, 2*np.pi, 2048, endpoint=False)
kx, ky = k_perp*np.cos(gamma), k_perp*np.sin(gamma)
components = np.exp(1j*rho[:, None]*kx)
cone_sum = np.mean(components, axis=1)
laplacian_sum = np.mean(-(kx**2+ky**2+k_z**2)*components, axis=1)
z_um = 1.0
propagated = cone_sum*np.exp(1j*k_z*z_um)
print('rho (um)   cone sum (real)   amplitude after propagation')
for radius, summed, later in zip(rho, cone_sum, propagated):
    print(f'{radius:5.1f}          {summed.real:+.6f}             {abs(later):.6f}')
print(f'Maximum normalized Helmholtz residual: {np.max(abs(laplacian_sum+k**2*cone_sum))/k**2:.2e}')
print(f'Maximum amplitude change: {np.max(np.abs(abs(propagated)-abs(cone_sum))):.2e}')

<details><summary>Answer</summary>

At the axis, all contributions align and their average is 1. The summed field has a zero Helmholtz residual up to rounding. Propagation multiplies every contribution by the same $e^{ik_z z}$, preserving the transverse amplitude.

</details>

### 5.8 Gaussian families and the paraxial approximation

The usual Gaussian beam and its Hermite–Gaussian and Laguerre–Gaussian families solve a reduced equation. Write $U=\psi(x,y,z)e^{ikz}$:

$$\partial_z^2U
=e^{ikz}\left(\partial_z^2\psi+2ik\partial_z\psi-k^2\psi\right),$$

$$\nabla^2U+k^2U
=e^{ikz}\left(\nabla_\perp^2\psi+\partial_z^2\psi+2ik\partial_z\psi\right)=0.$$

For a slowly varying envelope, $|\partial_z^2\psi|\ll|2k\partial_z\psi|$, giving

$$\boxed{\nabla_\perp^2\psi+2ik\partial_z\psi\simeq0.}$$

Here $\nabla_\perp^2=\partial_x^2+\partial_y^2$. A fundamental Gaussian solution has waist radius $w_0$ at $z=0$. With $\rho^2=x^2+y^2$,

$$\boxed{U(\rho,z)=A\frac{w_0}{w(z)}
\exp\!\left[-\frac{\rho^2}{w(z)^2}\right]
\exp\!\left\{i\left[kz+\frac{k\rho^2}{2R(z)}-\zeta(z)+\phi\right]\right\},}$$

$$z_R=\frac{\pi w_0^2}{\lambda},\qquad
w(z)=w_0\sqrt{1+(z/z_R)^2},$$

$$\frac1{R(z)}=\frac{z}{z^2+z_R^2},\qquad
\zeta(z)=\arctan(z/z_R).$$

The width grows from $w_0$, the wavefront curvature is $1/R$, and $\zeta$ is the Gouy phase. At the waist, $1/R=0$ and $\zeta=0$, giving $U(\rho,0)=A e^{-\rho^2/w_0^2}e^{i\phi}$. At one Rayleigh range,

$$w(z_R)=\sqrt2\,w_0,\qquad |U(0,z_R)|=\frac{A}{\sqrt2}.$$

Spreading therefore lowers the peak amplitude. A wider waist increases $z_R$ and reduces the far-field angular spread, $\theta_{\rm div}\simeq\lambda/(\pi w_0)$. The explorer uses waists of two to six wavelengths to keep this spread small.

This field solves the **paraxial** equation, not the full Helmholtz equation exactly. Hermite–Gaussian modes add Cartesian lobes; Laguerre–Gaussian modes add radial rings and, for nonzero azimuthal order, phase winding. The explorer below displays the fundamental Gaussian only. [Pampaloni and Enderlein, Gaussian-beam primer (2004)](https://arxiv.org/abs/physics/0410021).

**See these fields as beams.** A beam has a propagation axis and a transverse intensity profile. The Gaussian and ideal Bessel fields give two examples; a plane wave supplies a uniform reference with no finite width. To plot the beam, square the field magnitude. The phase factors have unit magnitude, so

$$I_{\rm G}(\rho,z)=|U_{\rm G}|^2=A^2\frac{w_0^2}{w(z)^2}\exp\!\left[-\frac{2\rho^2}{w(z)^2}\right],\qquad
I_{\rm B}(\rho,z)=|U_{\rm B}|^2=A^2J_0^2(k_\perp\rho).$$

These are relative scalar intensities using the fixed unit-amplitude reference from §1. At $z=z_R$, the Gaussian width is $\sqrt2\,w_0$ and its peak intensity is $A^2/2$; the ideal Bessel profile is unchanged with $z$.

**Compare the solution families:** use **Solution** to switch among the seven exact Helmholtz examples and **Gaussian beam (paraxial)**. **Field & amplitude** shows the signed real field on the left and $|U|$ along the stated line on the right. For the plane wave, Bessel beam, and Gaussian beam, switch **View** to **Beam intensity**: the three panels show an $xz$ side view, an $xy$ observation plane, and its intensity profile along $x$ at $y=0$. The cyan line marks that observation plane in the side view.

| Solution | What to look for |
| --- | --- |
| Plane wave | In **Field & amplitude**, crests travel along $+z$. **Beam intensity** is uniform in every plane; the plot boundaries are not beam edges. |
| Standing wave | Nodes stay fixed along $z$ as the field oscillates. |
| Spherical wave | Circular traces in $xz$ are slices of spherical wavefronts; radial amplitude along $+z$ falls as $1/r$. |
| Cylindrical wave | In $xz$, a line source produces slower radial spreading along $+z$ than the spherical wave. |
| Bessel beam | In **Beam intensity**, move **Observation z/λ**: the central core and rings retain their shape. Increase **Transverse k⊥/k** to narrow the core. |
| Spherical multipole | Vary **Angular degree ℓ** to compare angular nodes in $xz$; the profile follows $+z$. |
| Evanescent wave | In the $xz$ view, increasing **Evanescent q/k** shortens the amplitude decay into $+z$. |
| Gaussian beam (paraxial) | Select **Beam intensity** and move **Axial plane z/z_R** away from zero: the spot widens and its peak falls. The side view shows the waist and spreading together. |

**Beam intensity** uses physical distances in micrometres. The side view has different horizontal and vertical scales, as labelled; the transverse plane uses equal scales. Dashed lines mark the Gaussian radius $w(z)$, where intensity is $e^{-2}$ of that plane's peak, or the first Bessel zero. These are width guides, not hard edges. Intensity colours stay fixed from 0 to 1 across planes and families, so a reduced peak remains visible. Phase and model time are hidden because they do not change these monochromatic intensity maps.

In the Gaussian **Field & amplitude** view, transverse distances are scaled by $w_0$. Changing **Waist w₀/λ** changes the physical waist, Rayleigh range, and divergence; at fixed $z/z_R$, the profile plotted in waist units keeps its shape. **Model time t/T** changes the signed fields but leaves every amplitude profile unchanged. Wavelength and amplitude are shared controls; specialized controls appear only for the relevant family.

For display, the spherical field is $U=A\lambda e^{i(kr+\phi)}/r$, so its amplitude equals $A$ at $r=\lambda$; the cylindrical field is similarly normalized at $\rho=\lambda$. Their central regions $r<\lambda/2$ and $\rho<\lambda/2$ are masked only for drawing. Multipole $A$ is a coefficient, not a guaranteed peak value; Gaussian $A$ is the peak amplitude at its waist.

In [ ]:
show_scalar_theory('solutions')

<a id="amplitude-phase-comparison"></a>

### Reading the solutions as amplitude and phase

The families above share the representation introduced in Section 1:

$$U(\mathbf r)=\underbrace{A(\mathbf r)}_{\text{size of the local oscillation}}\,
\underbrace{e^{i\Phi(\mathbf r)}}_{\text{spatial phase}},\qquad A(\mathbf r)=|U(\mathbf r)|\geq0.$$

Amplitude describes how the oscillation's size varies across space; phase describes where each local oscillation lies in its cycle. Their roles separate when we calculate the squared magnitude:

$$|U|^2=(Ae^{i\Phi})(Ae^{-i\Phi})=A^2.$$

Here are four examples, with positive constant scales $A_0$ or $A_s$ and a common phase offset set to zero. The last column uses the relative scalar-intensity convention of Section 1.

| Field | Amplitude $A(\mathbf r)$ | Spatial phase $\Phi(\mathbf r)$ | Relative scalar intensity $\lvert U\rvert^2$ |
| --- | --- | --- | --- |
| Plane wave along $+z$ | $A_0$ | $kz$ | $A_0^2$ |
| Outgoing spherical wave, $r>0$ | $A_s/r$ | $kr$ | $A_s^2/r^2$ |
| Bessel beam along $+z$ | $A_0\lvert J_0(k_\perp\rho)\rvert$ | $k_z z+\delta(\rho)$ | $A_0^2J_0^2(k_\perp\rho)$ |
| Evanescent wave, $z\geq0$ | $A_0e^{-\alpha z}$ | $qx$ | $A_0^2e^{-2\alpha z}$ |

For the Bessel beam, $\rho=\sqrt{x^2+y^2}$ and $k_\perp^2+k_z^2=k^2$. For the evanescent wave, $q>k$ and $\alpha=\sqrt{q^2-k^2}$, as in Section 5.7.

**The Bessel sign belongs in the phase.** Since $J_0$ changes sign across each positive zero, $\delta(\rho)=0$ where $J_0>0$ and $\delta(\rho)=\pi$ where $J_0<0$: a negative factor can be written as $-\lvert J_0\rvert=\lvert J_0\rvert e^{i\pi}$. At a zero, the amplitude vanishes and phase is undefined. This keeps amplitude nonnegative throughout. [NIST DLMF, Bessel-function zeros](https://dlmf.nist.gov/10.21).

**Experimental connection:** for propagating light in the calibrated measurement geometry of Section 1, an intensity profile lets us infer amplitude ratios by taking square roots. The phase column requires additional information, such as interference with a reference. For the evanescent row, $|U|^2$ describes the decaying squared field magnitude; it is not a measure of power transported into $+z$.

The spatial variation of phase also connects these examples to wavevectors. [Notebook 1, Section 4](../1_Plane_Waves_and_Wavevectors.ipynb#local-phase-gradient) develops that connection in its optional gradient explanation.


### 5.9 Which solutions matter most for diffraction?

For scalar aperture diffraction, **plane waves and outgoing spherical waves are the main building blocks**. The other families serve particular geometries or illumination conditions.

| Family | Role in diffraction |
| --- | --- |
| **Plane waves** | **Angular spectrum method (ASM):** decompose the boundary field into components with different wavevectors, propagate each, and add them. |
| **Outgoing spherical waves** | **Rayleigh–Sommerfeld (RS):** build the propagation kernel from the outgoing spherical Green function (the ideal point-source field) and its normal derivative, taken perpendicular to the boundary. |
| **Evanescent waves** | Complete the angular spectrum when $k_x^2+k_y^2>k^2$; especially relevant close to fine boundary structure. |
| **Gaussian beams** | Model illumination of an aperture and paraxial beam propagation. |
| **Cylindrical waves** | Provide building blocks when the field is invariant along one direction, as for an ideal infinitely long slit. |
| **Bessel beams and spherical multipoles** | Suit particular symmetries and specialized diffraction or scattering problems. |
| **Standing waves** | Describe opposing contributions, commonly produced by reflections or in cavities. |

ASM and RS describe the same scalar propagation problem under compatible boundary and radiation conditions. [TU Delft, *Scalar Diffraction Optics*](https://interactivetextbooks.tudelft.nl/interactive-optics/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html), [*Scalar diffraction theory*](https://qiweb.tudelft.nl/aoi/scalardiffractiontheory/scalardiffractiontheory/).

For the next notebooks we choose **one plane wave**, then a small number of plane-wave contributions, to learn direction, phase, amplitude, and interference. This choice is useful for diffraction; it does not exclude the other solutions. A full ASM–RS treatment and numerical propagation come after those foundations.

## 6. Linearity lets us build new fields

**Preview:** solutions can be added. This is why simple plane waves can become building blocks for more complicated diffraction fields; Notebook 7 will develop the two-wave example.

Suppose $U_1$ and $U_2$ solve the same Helmholtz equation, and $a,b$ are constant complex weights. Then

$$\begin{aligned}
(\nabla^2+k^2)(aU_1+bU_2)
&=a(\nabla^2U_1+k^2U_1)+b(\nabla^2U_2+k^2U_2)\\
&=a\,0+b\,0=0.
\end{aligned}$$

Both components must use the **same $k$** for this argument. Different frequencies can be combined in the time-domain wave equation, but their sum cannot generally be represented by one spatial field $U$ with a single common time factor.

Linearity of fields does not make intensity linear:

$$|aU_1+bU_2|^2
=|aU_1|^2+|bU_2|^2
+2\operatorname{Re}\{aU_1(bU_2)^*\}.$$

### Small calculation 4: a solution and its interference pattern

Combine forward and backward waves with amplitudes 1 and 0.5. Predict the largest and smallest relative scalar intensities, then inspect the residual and intensity separately.

In [ ]:
k = np.pi
z = np.linspace(0, 2, 801)
amplitude1, amplitude2 = 1.0, 0.5
phase1_deg, phase2_deg = 0.0, 60.0
U1 = amplitude1*np.exp(1j*(k*z+np.deg2rad(phase1_deg)))
U2 = amplitude2*np.exp(1j*(-k*z+np.deg2rad(phase2_deg)))
U = U1+U2
d2U = -k**2*U1-k**2*U2
print(f'Maximum Helmholtz residual: {np.max(abs(d2U+k**2*U)):.2e}')
print(f'Separate-intensity sum: {amplitude1**2+amplitude2**2:.2f}')
print(f'Sampled intensity range: {np.min(abs(U)**2):.4f} to {np.max(abs(U)**2):.4f}')
print(f'Exact range: {(amplitude1-amplitude2)**2:.2f} to {(amplitude1+amplitude2)**2:.2f}')

<details><summary>Answer</summary>

The sum satisfies Helmholtz up to rounding. Its relative intensity varies between $(1-0.5)^2=0.25$ and $(1+0.5)^2=2.25$, around the separate-intensity sum $1.25$. A sampled grid may miss an exact extremum slightly.

</details>

## 7. Which allowed solution describes the experiment?

**Preview:** the equation alone does not select the field. The example below shows why the boundary value and an outgoing-wave condition are both needed; its coefficient algebra is optional on a first reading.

Helmholtz supplies a family of solutions. The source, boundary data, and propagation or radiation condition select the physical member.

In one dimension,

$$U''+k^2U=0
\quad\Rightarrow\quad
U(z)=B_+e^{ikz}+B_-e^{-ikz}.$$

The condition $U(0)=U_0$ gives only

$$B_++B_-=U_0.$$

It does not determine two independent coefficients. Their derivative provides another relation:

$$U'(0)=ik(B_+-B_-),$$

$$B_+=\frac12\left[U_0+\frac{U'(0)}{ik}\right],
\qquad
B_-=\frac12\left[U_0-\frac{U'(0)}{ik}\right].$$

The explorer uses a simple family sharing the same boundary value:

$$\begin{aligned}
U(z)
&=e^{i\phi}\left[(1-b)e^{ikz}+b e^{-ikz}\right]\\
&=e^{i\phi}\left[\cos(kz)+i(1-2b)\sin(kz)\right],\\
U(0)&=e^{i\phi},\qquad U'(0)=ik(1-2b)e^{i\phi}.
\end{aligned}$$

**Try:** set **Backward fraction b** to 0, 0.5, and 1. These give forward, standing, and backward fields with the same $U(0)$. The left panel plots the real and imaginary parts of $U$; at $\phi=0$, their real parts coincide, but their imaginary parts differ. The right panel shows the real field across position and time: travelling crests reverse their tilt, while standing nodes remain fixed. **Model time t/T** selects a horizontal time slice on that map.

In [ ]:
show_scalar_theory('boundary')

If the region is $z>0$ and no wave arrives from its far end, the outgoing choice is $B_-=0$. Then $U(z)=U_0e^{ikz}$. The boundary value becomes sufficient **together with** this directional condition.

For curved waves, an outgoing radiation condition similarly distinguishes the outward spherical solution from the inward one. We should not choose a function only because its picture resembles an experiment; its boundary and source conditions must also fit.

## 8. Use the theory to describe diffraction

An ideal thin aperture assigns a boundary field in the plane $z=0$:

$$U(\xi,0)=t(\xi)U_{\rm incident}(\xi,0).$$

For a uniformly illuminated long slit of width $a$, take the field uniform along $y$. We suppress that unchanged coordinate and study the $x$ profile across the slit:

$$U(\xi,0)=
\begin{cases}
U_0,&|\xi|<a/2,\\
0,&|\xi|>a/2.
\end{cases}$$

This is an ideal scalar mask assignment. It does not solve the electromagnetic scattering at the slit edges. The propagation task is to find an outgoing field in $z>0$ that satisfies Helmholtz and the assigned boundary data, then calculate its intensity.

### 8.1 Optional derivation: a controlled far-field example

At a screen position $X$ and distance $L$, a contribution from aperture position $\xi$ has distance

$$R=\sqrt{L^2+(X-\xi)^2}
\simeq L+\frac{(X-\xi)^2}{2L}
=L+\frac{X^2}{2L}-\frac{X\xi}{L}+\frac{\xi^2}{2L}.$$

The paraxial approximation requires $|X-\xi|\ll L$. In the **Fraunhofer limit**, the aperture's quadratic phase is also small:

$$\frac{k\xi^2}{2L}\ll1,\qquad
\frac{a^2}{\lambda L}\ll1.$$

Dropping that quadratic aperture term leaves the relative phase $-kX\xi/L$. The remaining propagation factors have the same magnitude across this paraxial screen. Retaining the quadratic screen phase, the field is

$$U_F(X,L)\propto e^{ikX^2/(2L)}U_0\int_{-a/2}^{a/2}e^{-i(kX/L)\xi}\,d\xi.$$

This integral is a far-field propagation approximation, not the general near-field solution. [TU Delft, *Scalar Diffraction Optics*, Fresnel and Fraunhofer approximations](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html).

### 8.2 Evaluate the slit integral

Let $q=kX/L$ and $\beta=qa/2$. For $q\ne0$,

$$\begin{aligned}
\int_{-a/2}^{a/2}e^{-iq\xi}\,d\xi
&=\left[\frac{e^{-iq\xi}}{-iq}\right]_{-a/2}^{a/2}\\
&=\frac{e^{-iqa/2}-e^{iqa/2}}{-iq}\\
&=\frac{-2i\sin(qa/2)}{-iq}
=a\frac{\sin\beta}{\beta},
\qquad
\beta=\frac{\pi aX}{\lambda L}.
\end{aligned}$$

At $X=0$, the integral is directly $a$, agreeing with $\lim_{\beta\to0}\sin\beta/\beta=1$. Squaring the amplitude and normalizing to the centre gives

$$\boxed{\frac{I(X)}{I(0)}
=\left(\frac{\sin\beta}{\beta}\right)^2.}$$

The first zeros occur at $\beta=\pm\pi$:

$$\frac{\pi aX_{\rm zero}}{\lambda L}=\pm\pi
\quad\Rightarrow\quad
X_{\rm zero}=\pm\frac{\lambda L}{a}.$$

The central bright region therefore has width $2\lambda L/a$. Narrowing the slit broadens the pattern: the changed aperture field produces a different spatial distribution after propagation.

**Graphical check:** the left panel shows the aperture field; the right shows the normalized screen intensity. At reset, $\lambda=0.6\,\mu\mathrm m$, $a=20\,\mu\mathrm m$, and $L=1000\,\mathrm{mm}$, giving first zeros at $\pm30\,\mathrm{mm}$. Double the slit width to $40\,\mu\mathrm m$ and compare. Each curve is normalized to its own central intensity, so these graphs compare shape rather than absolute transmitted power.

**Read the experiment backwards:** measure the distance $W$ between the first dark minima
on opposite sides of the central maximum and use a known wavelength and screen distance. Within this
far-field model,

$$W=\frac{2\lambda L}{a}\quad\Longrightarrow\quad
a=\frac{2\lambda L}{W}.$$

The screen supplies a width; the model converts it into an inferred slit width.
For the reset example, $W=60\,\mathrm{mm}$ gives $a=20\,\mu\mathrm m$.


In [ ]:
show_scalar_theory('diffraction')

### Small calculation 5: add aperture contributions before squaring

The midpoint sum below approximates the slit integral divided by $a$. Compare its complex amplitude with the analytic answer at the centre, halfway to the first zero, at the first zero, and beyond it. Here $\xi$ is in micrometres and $X,L$ are both in millimetres, so $X/L$ is dimensionless.

In [ ]:
lambda_um, a_um, L_mm = 0.6, 20.0, 1000.0
k = 2*np.pi/lambda_um
first_zero_mm = lambda_um*L_mm/a_um
X_mm = first_zero_mm*np.array([0.0, 0.5, 1.0, 1.5])
sample_count = 12000
xi_um = -a_um/2+(np.arange(sample_count)+0.5)*a_um/sample_count
amplitude_sum = np.mean(np.exp(-1j*k*(X_mm[:, None]/L_mm)*xi_um), axis=1)
beta = np.pi*a_um*X_mm/(lambda_um*L_mm)
# NumPy sinc(s) means sin(pi*s)/(pi*s), including its value 1 at s=0.
analytic_amplitude = np.sinc(beta/np.pi)
print('X (mm)   summed amplitude   analytic amplitude   normalized intensity')
for X, summed, exact in zip(X_mm, amplitude_sum, analytic_amplitude):
    print(f'{X:6.1f}      {summed.real:+.6f}            {exact:+.6f}               {abs(summed)**2:.6f}')
print(f'Maximum complex-amplitude mismatch: {np.max(abs(amplitude_sum-analytic_amplitude)):.2e}')

<details><summary>Answer</summary>

The normalized amplitudes are $1,\,2/\pi,\,0,\,-2/(3\pi)$, giving intensities approximately $1,\,0.4053,\,0,\,0.0450$. A negative amplitude is a phase reversal, not negative intensity. At the first zero, nonzero aperture contributions cancel in the field sum.

</details>

### What the aperture preview tells us

Changing the aperture changes the boundary field $U$, which changes the propagated field and its intensity pattern. This is the scalar diffraction task. The slit result answers one restricted far-field question; general propagation methods can wait until the plane-wave building blocks and their superposition are familiar.

## Preview checkpoint

1. What quantity do we seek in a scalar diffraction calculation?
2. Does the scalar Helmholtz equation allow only plane waves?
3. Why choose plane waves for the next notebooks?

<details><summary>Answers to the preview checkpoint</summary>

1. The complex scalar field $U$: it gives amplitude and spatial phase, and its squared magnitude gives the relative intensity in our convention.
2. No. Several exact families exist, and approximate families such as paraxial Gaussian beams are useful when their assumptions hold. Sources and boundary conditions decide which field describes an experiment.
3. Their simple spatial phase makes direction and wavelength easy to connect, and sums of plane waves provide a useful description of diffraction fields.

</details>

### Optional checkpoint: revisit the detailed calculations

These questions check the derivations above; they are not prerequisites for moving to Notebook 1.

1. Does “scalar” require plane wavefronts?
2. What does the one-dimensional wave equation require of $q$ in $A\cos(qx-\omega t)$?
3. Where does the positive $k^2U$ term in Helmholtz come from?
4. At fixed frequency, what happens to wavelength when the refractive index doubles?
5. Why is a nonzero residual at even one source-free point enough to reject a proposed exact solution there?
6. How can a standing wave satisfy Helmholtz? Does a spherical wave satisfy it at its source?
7. Why does a two-dimensional picture of spherical waves not justify using the two-dimensional radial Laplacian?
8. Do $U(0)$ and Helmholtz alone specify the direction of a one-dimensional field?
9. If two fields satisfy the same Helmholtz equation, do their intensities necessarily add?
10. What approximation produces the slit sinc pattern, and how does its central width change when the slit width doubles?
11. Why can an ideal Bessel beam retain its transverse profile, and why is a laboratory version only approximate?
12. How do spherical multipoles add to the spherical-wave family, and which radial choice is regular at the origin?
13. Are evanescent fields and the usual Gaussian beam formulas both exact Helmholtz solutions?

<details><summary>Answers</summary>

1. No. Scalar describes the field quantity; allowed wavefront shapes depend on the solution.
2. $q^2=\omega^2/v^2$. Either sign of $q$ is possible; the sign selects the travelling direction.
3. Two time derivatives of $e^{-i\omega t}$ give $-\omega^2e^{-i\omega t}$. The minus sign in the wave equation then produces $+\omega^2U/v^2=+k^2U$.
4. It halves: $\lambda=\lambda_0/n$, while the frequency remains fixed.
5. The differential equation must hold locally throughout the claimed source-free domain. Approximate solutions or numerical residuals require a separate accuracy assessment.
6. It is a sum of two same-$k$ solutions. The ideal spherical form solves the homogeneous equation only for $r>0$; its origin is a source singularity.
7. The picture is a slice of a three-dimensional field. Its radial Laplacian includes $2U'/r$; an axis-invariant cylindrical problem includes $U'/\rho$.
8. No: both coefficients obey only $B_++B_-=U(0)$. A derivative condition or a suitable directional condition supplies the missing information.
9. No. Fields add linearly, but squaring their sum generally introduces a cross term.
10. The paraxial Fraunhofer approximation neglects quadratic phase across the aperture. The central width $2\lambda L/a$ halves when $a$ doubles, with wavelength and distance fixed.
11. Its cone components share the same $k_z$, so propagation changes only their common phase. The ideal field has infinite extent and power; finite apertures limit the approximation.
12. They include angular patterns $Y_{\ell m}$. The $j_\ell$ radial choice is regular; outgoing $h_\ell^{(1)}$ is singular at the origin.
13. The displayed evanescent field solves full Helmholtz exactly in its half-space. Standard Gaussian, Hermite–Gaussian, and Laguerre–Gaussian formulas use the paraxial approximation.

</details>

## Next: understand one plane-wave building block

We keep the same central quantity $U$ and select the plane-wave form

$$U(\mathbf r)=A e^{i(\mathbf k\cdot\mathbf r+\phi_0)},\qquad
u(\mathbf r,t)=A\cos(\mathbf k\cdot\mathbf r-\omega t+\phi_0).$$

**How do wavelength and direction appear in this field?** [Notebook 1: Plane waves and wavevectors](../1_Plane_Waves_and_Wavevectors.ipynb) picks up that question by following the real oscillation at one point, then the wavefronts across space. The solution families and diffraction example above remain a map to return to as the details become familiar.

## References and conventions

- A. P. Konijnenberg, A. J. L. Adam, and H. P. Urbach, [*Basic Electromagnetic and Wave Optics*, TU Delft](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap1_Basics/Basics_2022_01Clean.html): scalar wave equation, harmonic representation, Helmholtz equation, plane and spherical waves. This notebook always uses $\lambda$ for wavelength **in the medium**.
- F. J. W. Olver et al., eds., [NIST *Digital Library of Mathematical Functions*, §10.73](https://dlmf.nist.gov/10.73): cylindrical and spherical wave-equation solutions.
- J. Durnin, [*Exact solutions for nondiffracting beams. I. The scalar theory*, JOSA A 4, 651–654 (1987)](https://www.durnin.info/durnin_josa_1987.pdf): ideal Bessel beams and finite-aperture realizations.
- F. Pampaloni and J. Enderlein, [*Gaussian, Hermite-Gaussian, and Laguerre-Gaussian beams: A primer* (2004)](https://arxiv.org/abs/physics/0410021): these beam families within the paraxial approximation.
- TU Delft, [*Scalar diffraction theory*, Advanced Optical Imaging](https://qiweb.tudelft.nl/aoi/scalardiffractiontheory/scalardiffractiontheory/): boundary-integral and angular-spectrum descriptions.
- A. P. Konijnenberg, A. J. L. Adam, and H. P. Urbach, [*Scalar Diffraction Optics*, TU Delft](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html): propagation representations and the Fresnel/Fraunhofer approximations.

The derivative checks and worked numerical examples here are constructed directly from the displayed equations. The interactive pictures illustrate prescribed scalar models, not a full electromagnetic aperture simulation.